In [9]:
using DataFrames, Distributions, CSV, Turing, Optim, StatsBase, StatsFuns, Random

Random.seed!(1234)

# Read the vitamin D dataset
data = CSV.read("../../data/vitd/nhanes.csv", DataFrame)
data[!, :"RIAGENDR"] = data[!, :"RIAGENDR"] .- 1
#data[!, :"DPQ020"] = data[!, :"DPQ020"] .+ 1
data[!, :"DPQ020"] = data[!, :"DPQ020"]

# Look at the first few rows of the data
first(data, 5)

Row,BMXBMI,RIAGENDR,RIDAGEYR,LBXVIDMS,INDFMMPI,DPQ020,SMQ040
,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,27.8,0.0,62.0,76.1,4.14,0.0,3.0
2,30.8,0.0,53.0,56.5,0.0,0.0,1.0
3,28.8,0.0,78.0,87.5,1.81,0.0,3.0
4,28.0,0.0,22.0,47.2,2.98,0.0,2.0
5,27.6,0.0,46.0,44.5,1.73,0.0,1.0


In [10]:
#= Convert categorical data to integers =#

data[!, :"RIAGENDR"] = convert(Array{Int}, data[!, :"RIAGENDR"])
data[!, :"DPQ020"] = convert(Array{Int}, data[!, "DPQ020"])
data[!, :"SMQ040"] = convert(Array{Int}, data[!, :"SMQ040"]);

# Get variables
bmi = data[!, :"BMXBMI"]
gender = data[!, :"RIAGENDR"]
age = data[!, :"RIDAGEYR"]
vitd = data[!, :"LBXVIDMS"]
poverty = data[!, :"INDFMMPI"]
depression = data[!, :"DPQ020"]
smoking = data[!, :"SMQ040"];
replace!(data.:"DPQ020", 0 => 0, 1 => 0, 2 => 1, 3 => 1);

In [11]:
# Turing Model OPTIM

@model function depression_model(bmi, gender, age, vitd, poverty, depression, smoking)
    # Define the length 
    n = length(min(length(bmi), length(gender), length(age)))

    # Define the priors
    μ_bmi ~ Normal(3.371267719188432, 0.5)
    β_age_bmi ~ Normal(0, 1)
    β_smoking_bmi ~ Normal(0, 1)
    
    σ_bmi ~ Exponential(0.22)
    α_age_1 ~ Normal(32, 1)
    α_age_2 ~ Normal(51, 1)
    α_vitd ~ Normal(4.07522197652439, 0.1)
    β_smoking_vitd ~ Normal(0, 1)
    β_age_vitd ~ Normal(0, 1)
    β_gender_vitd ~ Normal(0, 1)
    β_bmi_vitd ~ Normal(0, 1)
    σ_vitd ~ Exponential(0.42)

    
    α_depression ~ Normal(0, 1)
    β_vitd_depression ~ Normal(0, 1)
    β_bmi_depression ~ Normal(0, 1)
    β_age_depression_1 ~ Normal(0,0.1)
    β_age_depression_2 ~ Normal(0,0.1)
    β_age_depression_3 ~ Normal(0,0.1)
    β_poverty_depression ~ Normal(0, 0.1)
    β_gender_depression ~ Normal(0, 0.1)
    β_smoking_depression ~ Normal(0, 0.1)

    # θ_1 ~ Normal(1,3)
    # θ_2 ~ Normal(2,3)

    for i in 1:n
        dist1_age = Normal(α_age_1, 7.2)
        dist2_age = Normal(α_age_2, 8.7)
        dist1_pov = Normal(1, 2)
        dist2_pov = Normal(5, 0.001)
        gender[i] ~ Bernoulli(0.4)
        age[i] ~ MixtureModel([dist1_age, dist2_age], [0.4, 0.6])
        smoking[i] ~ Categorical([0.35, 0.12, 0.53]) 
        poverty[i] ~ MixtureModel([dist1_pov, dist2_pov], [0.5, 0.5])

        bmi[i] ~ LogNormal(μ_bmi + β_age_bmi * age[i] + β_smoking_bmi * smoking[i], σ_bmi)

        vitd[i] ~ LogNormal(α_vitd + β_smoking_vitd * smoking[i] + β_age_vitd * age[i]
        + β_gender_vitd * gender[i] + β_bmi_vitd * bmi[i], σ_vitd)

        if age[i] < 30
            linear_predictor = α_depression + vitd[i] * β_vitd_depression +
            bmi[i] * β_bmi_depression + age[i] * β_age_depression_1 +
            poverty[i] * β_poverty_depression + gender[i] * β_gender_depression +
           smoking[i] * β_smoking_depression

        elseif age[i] >= 30 && age[i] < 50

            linear_predictor = α_depression + vitd[i] * β_vitd_depression +
            bmi[i] * β_bmi_depression + age[i] * β_age_depression_2 +
            poverty[i] * β_poverty_depression + gender[i] * β_gender_depression +
           smoking[i] * β_smoking_depression
        
        else

            linear_predictor = α_depression + vitd[i] * β_vitd_depression +
            bmi[i] * β_bmi_depression + age[i] * β_age_depression_3 +
            poverty[i] * β_poverty_depression + gender[i] * β_gender_depression +
           smoking[i] * β_smoking_depression
        end
        
        #depression[i] ~ Categorical(softmax([0, linear_predictor, linear_predictor + θ_1, linear_predictor + θ_2]))
        p_depression = logistic(linear_predictor)
        depression[i] ~ Bernoulli(p_depression)

    end
end


depression_model (generic function with 2 methods)

In [14]:
Random.seed!(1234)
model_optim = depression_model(bmi, gender, age, vitd, poverty, depression, smoking)
map_estimate = optimize(model_optim, MAP(), NelderMead(), Optim.Options(iterations=1000000, show_trace=true, show_every=1000))

Iter     Function value    √(Σ(yᵢ-ȳ)²)/n 
------   --------------    --------------
     0     6.930161e+04     6.370856e+04
 * time: 0.00036406517028808594
  1000     1.784254e+01     6.738122e-02
 * time: 0.14153718948364258
  2000     1.399687e+01     1.660528e-02
 * time: 0.2999420166015625
  3000     1.342884e+01     4.490470e-03
 * time: 0.4044780731201172
  4000     1.155576e+01     2.796868e-03
 * time: 0.5174531936645508
  5000     1.045564e+01     2.279676e-02
 * time: 0.616873025894165
  6000     9.564553e+00     2.441332e-03
 * time: 0.725675106048584
  7000     7.485460e+00     1.860265e-02
 * time: 0.8209421634674072
  8000     7.181006e+00     1.749303e-05
 * time: 0.9247970581054688
  9000     7.135825e+00     1.532249e-03
 * time: 1.0282011032104492
 10000     7.090685e+00     5.577862e-05
 * time: 1.1338520050048828
 11000     5.465065e+00     6.196029e-02
 * time: 1.242894172668457
 12000     1.872166e+00     4.508670e-03
 * time: 1.344128131866455
 13000     1.6608

InterruptException: InterruptException:

In [13]:
io = IOBuffer()
map_names = names(map_estimate.values)[1]
for i in 1:length(map_estimate.values)
    map_name = map_names[i]
    value = map_estimate.values[i]
    println(io, "$map_name = $value")
end
to_text = String(take!(io))
file_path = "../../results/vitd/turing_optim_map_alt.txt"

open(file_path, "w") do file
    write(file, to_text)
end

766